# CHW Data Pipeline — Live Demo
**Mercy Chepngeno | Digital Public Infrastructure Engineer | Nairobi, Kenya**

GitHub: https://github.com/chep-collab/chw-data-pipeline

---

I built this pipeline while working on a community registry in Zambia. We kept running into the same data problems on every deployment — mixed date formats, field names that changed between tools, duplicates from sync conflicts, missing values nobody could explain. After the third time cleaning the same mess manually, I decided to build something that handled it properly.

This notebook walks through the full pipeline:
- Missing values from interrupted mobile data collection
- Inconsistent date formats across country configurations
- Duplicate records from offline sync conflicts
- Field names that mean different things in Kenya vs Zambia

**Countries in this demo:** Kenya and Zambia

---

In [ ]:
# Install dependencies
!pip install pandas numpy pyyaml -q
print('Dependencies ready')

In [ ]:
import pandas as pd
import numpy as np
import json
import hashlib
import uuid
from datetime import datetime
from typing import Dict, List, Optional, Tuple
from enum import Enum
import logging
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)
print('Imports ready')

## Step 1 — Generate Realistic Messy CHW Data

This simulates the kind of dataset you actually receive from KoboToolbox, ODK, or CommCare when two countries are running the same programme with different tool configurations.

In [ ]:
import random
random.seed(42)
np.random.seed(42)

# The date format issue was the one that hurt us most in Zambia.
# KoboToolbox exports ISO by default, but when the team switched tools
# mid-project we ended up with three different formats in the same column.
# This function recreates that exact scenario.

def generate_messy_chw_data(n=300):
    counties_ke = ['Turkana', 'Marsabit', 'Garissa', 'Wajir', 'Nairobi']
    counties_zm = ['Lusaka', 'Copperbelt', 'Northern', 'Eastern']
    visit_types = ['ANC Visit', 'Child Health', 'Family Planning', 'TB Screening', 'Malaria Testing']
    outcomes = ['Completed', 'Referred', 'Missed', None]

    rows = []
    for i in range(n):
        country = random.choice(['KE', 'ZM'])
        month = random.randint(1, 12)
        day = random.randint(1, 28)

        # Kenya used ISO, Zambia used DD/MM/YYYY or DD-MM-YYYY depending on the enumerator
        if country == 'KE':
            date_str = f'2024-{month:02d}-{day:02d}'
        else:
            date_str = random.choice([
                f'{day:02d}/{month:02d}/2024',
                f'{day:02d}-{month:02d}-2024'
            ])

        # Kenya and Zambia used completely different field names for the same data.
        # This is the multi-country schema problem in its most frustrating form.
        if country == 'KE':
            row = {
                'hh_id': f'HH-{random.randint(1000,9999)}',
                'visit_dt': date_str,
                'worker_id': f'CHW-{random.randint(100,999)}',
                'county': random.choice(counties_ke),
                'visit_category': random.choice(visit_types),
                'result': random.choice(outcomes),
                'age': random.randint(15,75) if random.random() > 0.1 else None,
                'referred': random.choice([True, False, None]),
                'country': 'KE'
            }
        else:
            row = {
                'household_code': f'HH-{random.randint(1000,9999)}',
                'date_of_visit': date_str,
                'enumerator_id': f'CHW-{random.randint(100,999)}',
                'province': random.choice(counties_zm),
                'service_type': random.choice(visit_types),
                'service_outcome': random.choice(outcomes),
                'age_at_visit': random.randint(15,75) if random.random() > 0.08 else None,
                'referral_status': random.choice([True, False, None]),
                'country': 'ZM'
            }
        rows.append(row)

    # Inject duplicates to simulate offline sync conflicts
    # These are not errors — they happen when a CHW loses connection mid-sync
    # and the device retries on reconnect without knowing the first attempt went through
    duplicates = random.sample(rows[:50], 15)
    rows.extend(duplicates)
    random.shuffle(rows)

    return pd.DataFrame(rows)

raw_df = generate_messy_chw_data(300)

print(f'Raw dataset: {len(raw_df)} rows, {len(raw_df.columns)} columns')
print(f'Countries: Kenya and Zambia mixed together')
print(f'Date formats: mixed — ISO, DD/MM/YYYY, DD-MM-YYYY')
print(f'Duplicates injected: ~15 records from sync conflicts')
print(f'Missing values: ~8-10% across key fields')
print('\nFirst 5 rows of raw data:')
raw_df.head()

## Step 2 — Profile the Dataset

I always audit before cleaning. The temptation is to jump straight into fixing things, but if you don't understand why the data is messy first, you end up making assumptions that cause worse problems downstream.

In [ ]:
def profile_dataset(df, label='dataset'):
    print(f'\n--- PROFILE: {label} ---')
    print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')

    total_cells = df.size
    null_cells = df.isnull().sum().sum()
    completeness = round((1 - null_cells/total_cells)*100, 1)
    print(f'Completeness: {completeness}%')
    print(f'Exact duplicates: {df.duplicated().sum()}')

    print(f'\nNull rates per field:')
    for col in df.columns:
        null_rate = df[col].isnull().mean()*100
        if null_rate > 0:
            flag = ' <- worth checking' if null_rate > 10 else ''
            print(f'  {col:<25} {null_rate:.1f}%{flag}')

    print(f'\nDate field samples (checking for format inconsistency):')
    date_cols = [c for c in df.columns if any(k in c.lower() for k in ['date','visit_dt','dt'])]
    for col in date_cols:
        samples = df[col].dropna().head(5).tolist()
        print(f'  {col}: {samples}')

    return completeness

score = profile_dataset(raw_df, 'raw CHW data')

## Step 3 — Country Schema Mapping

This is the part that always needs a conversation with the country teams before you write a single line of code. You cannot resolve field name conflicts from a laptop — you need to know what the field actually means in each context.

In [ ]:
# Built from conversations with the Kenya and Zambia programme teams.
# Same data, different names — happens every time two countries run the same programme independently.

COUNTRY_MAPPINGS = {
    'KE': {
        'hh_id': 'household_id',
        'visit_dt': 'visit_date',
        'worker_id': 'chw_id',
        'visit_category': 'visit_type',
        'result': 'outcome',
        'age': 'patient_age',
        'referred': 'referral_made',
    },
    'ZM': {
        'household_code': 'household_id',
        'date_of_visit': 'visit_date',
        'enumerator_id': 'chw_id',
        'province': 'county',
        'service_type': 'visit_type',
        'service_outcome': 'outcome',
        'age_at_visit': 'patient_age',
        'referral_status': 'referral_made',
    }
}

def apply_country_mapping(df):
    frames = []
    for country, group in df.groupby('country'):
        mapping = COUNTRY_MAPPINGS.get(country, {})
        group = group.rename(columns={k: v for k, v in mapping.items() if k in group.columns})
        frames.append(group)
    return pd.concat(frames, ignore_index=True)

mapped_df = apply_country_mapping(raw_df)
print('Country mapping applied')
print(f'Columns now: {list(mapped_df.columns)}')
print(f'\nKE: hh_id -> household_id, visit_dt -> visit_date, result -> outcome')
print(f'ZM: household_code -> household_id, date_of_visit -> visit_date, service_outcome -> outcome')

## Step 4 — Clean the Data

Two rules I follow here: standardise dates per-value not per-column (because mixed formats exist in the same column), and flag missing values instead of imputing them. Imputing health data introduces bias you can't see — a missing value usually means something.

In [ ]:
DATE_FORMATS = [
    '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y',
    '%d-%m-%Y', '%Y%m%d', '%d.%m.%Y'
]

def parse_date(val):
    if pd.isna(val):
        return val
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(str(val).strip(), fmt).strftime('%Y-%m-%d')
        except ValueError:
            continue
    # If nothing matches, keep the original and let the flag catch it
    return val

def clean_dataset(df):
    df = df.copy()
    audit = []

    # Standardise dates
    if 'visit_date' in df.columns:
        before = df['visit_date'].head(3).tolist()
        df['visit_date'] = df['visit_date'].apply(parse_date)
        after = df['visit_date'].head(3).tolist()
        audit.append(f'visit_date standardised to ISO 8601 | before: {before[:2]} | after: {after[:2]}')

    # Flag missing values — not impute
    df['__quality_flags__'] = ''
    for col in ['outcome', 'patient_age', 'referral_made']:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            if null_count > 0:
                df.loc[df[col].isnull(), '__quality_flags__'] += f'MISSING_{col.upper()}|'
                audit.append(f'{col}: {null_count} missing values flagged')

    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

    return df, audit

cleaned_df, audit_log = clean_dataset(mapped_df)

print('Cleaning done')
print('\nAudit log:')
for entry in audit_log:
    print(f'  {entry}')

flagged = (cleaned_df['__quality_flags__'] != '').sum()
print(f'\n{flagged} records flagged for review')
cleaned_df[cleaned_df['__quality_flags__'] != ''][['household_id','visit_date','outcome','__quality_flags__']].head(5)

## Step 5 — Deduplicate

This one took me a while to get right. I kept dropping duplicates until I realised two CHWs had visited the same household on the same day and both records were real. Now I flag and classify instead of drop.

In [ ]:
def deduplicate(df):
    df = df.copy()
    df['__duplicate_status__'] = 'PRIMARY'
    df['__duplicate_reason__'] = ''

    data_cols = [c for c in df.columns if not c.startswith('__')]
    exact_mask = df.duplicated(subset=data_cols, keep='first')
    df.loc[exact_mask, '__duplicate_status__'] = 'SECONDARY'
    df.loc[exact_mask, '__duplicate_reason__'] = 'EXACT_DUPLICATE'

    exact_count = exact_mask.sum()
    primary_count = len(df) - exact_count

    print(f'Deduplication done')
    print(f'  Total:     {len(df):,}')
    print(f'  Primary:   {primary_count:,}')
    print(f'  Secondary: {exact_count:,} (kept with flag — not deleted)')
    print(f'\n  Duplicates in CHW data are often legitimate — two workers,\n  same household, same day. Silent deletion loses real visits.')

    return df

final_df = deduplicate(cleaned_df)
final_df[final_df['__duplicate_status__'] == 'SECONDARY'][['household_id','visit_date','__duplicate_status__','__duplicate_reason__']].head(5)

## Step 6 — Summary & Output

In [ ]:
import matplotlib.pyplot as plt

primary_df = final_df[final_df['__duplicate_status__'] == 'PRIMARY'].copy()
completeness_after = round((1 - primary_df.isnull().mean().mean()) * 100, 1)
flagged_records = (primary_df['__quality_flags__'] != '').sum()

print('=' * 55)
print('PIPELINE SUMMARY')
print('=' * 55)
print(f'Input:             {len(raw_df):,} records')
print(f'Output (primary):  {len(primary_df):,} records')
print(f'Duplicates:        {len(raw_df) - len(primary_df):,} (preserved as SECONDARY)')
print(f'Completeness:      ~88% -> {completeness_after}% (flagged, not imputed)')
print(f'Quality flags:     {flagged_records:,} records need review')
print(f'Countries:         Kenya, Zambia')
print(f'Dates unified:     ISO 8601')
print('=' * 55)
print('Dataset is AI-ready. Audit trail preserved throughout.')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('CHW Data Pipeline — Output Report', fontsize=13, fontweight='bold')

country_counts = primary_df['country'].value_counts()
axes[0].bar(country_counts.index, country_counts.values, color=['#0F6E56', '#5DCAA5'])
axes[0].set_title('Records by Country', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].spines[['top', 'right']].set_visible(False)

flag_counts = {'Clean': len(primary_df) - flagged_records, 'Flagged': flagged_records}
axes[1].pie(flag_counts.values(), labels=flag_counts.keys(),
            colors=['#0F6E56', '#F9A825'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Data Quality', fontweight='bold')

if 'visit_type' in primary_df.columns:
    vtype = primary_df['visit_type'].value_counts().head(5)
    axes[2].barh(vtype.index, vtype.values, color='#0F6E56')
    axes[2].set_title('Visit Types', fontweight='bold')
    axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('pipeline_output.png', dpi=150, bbox_inches='tight')
plt.show()

---

## About

This reflects patterns from a community registry deployment covering 250,000+ households in Zambia — where the data quality problems in this notebook were real, not simulated.

GitHub: https://github.com/chep-collab/chw-data-pipeline

Mercy Chepngeno | Nairobi, Kenya | mercychepngeno582@gmail.com